# Diel δ¹³C(DIC) Model Calibration with PHREEQC (24 h)

**Objective.** Calibrate a diel biogeochemical model to observed δ¹³C(DIC) time series by optimizing:
- **e = ER:GPP** (dimensionless ecosystem respiration fraction of GPP)
- **k600** (CO₂ gas transfer velocity normalized to Sc=600; cm/hr)

The model couples:
1) Metabolism (sinusoidal daytime GPP; constant ER)
2) Air–water CO₂ exchange (k600 + Schmidt correction)
3) Carbonate equilibrium (PHREEQC inline `RunString`) to obtain pH and carbon speciation
4) δ¹³C(DIC) evolution via explicit ¹²C/¹³C mass balance + fractionation

Outputs:
- Parameter sweep contour (goodness-of-fit)
- Least-squares optimum + uncertainty propagation
- Diel plots (pH, pCO₂, GPP/ER, CO₂ flux, speciation, δ¹³C(DIC) modeled vs observed)
- Publication-quality SVG exports and full CSV export
- Methods paragraph (150–250 words)

## Installation (local)

Recommended (conda):
```bash
conda create -n diel13c python=3.11 -y
conda activate diel13c
pip install numpy pandas matplotlib scipy ipywidgets astral phreeqpy

In [ ]:

---

### 3) Imports + reproducibility (Code cell)

```python
import os
import sys
import math
import time
from dataclasses import dataclass
from datetime import datetime, date, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares
from scipy.stats import multivariate_normal

# Solar timing (astral)
from astral import LocationInfo
from astral.sun import sun

# Widgets
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# --- Reproducibility ---
RNG = np.random.default_rng(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# =========================
# CONFIGURATION (EDIT HERE)
# =========================

# Field site / date
LATITUDE = 45.0
LONGITUDE = -93.0
SITE_NAME = "Shallow Lake"
DATE0 = date(2024, 7, 15)
TIMEZONE_NAME = "UTC"  # keep UTC for reproducibility; change if needed

# Lake geometry
LAKE_DEPTH_M = 1.0
LAKE_VOLUME_M3 = 26_000.0
LAKE_AREA_M2 = LAKE_VOLUME_M3 / LAKE_DEPTH_M
LAKE_LITERS = LAKE_VOLUME_M3 * 1000.0

# Physical forcing
WATER_TEMPERATURE_C = 25.0
WIND_SPEED_10M = 3.0
ATM_PCO2_UATM = 420.0
ATM_D13C_CO2 = -8.0  # per mil VPDB-like convention

# Metabolism forcing
GPP_MAX_MG_O2_L_HR = 20.0   # peak at solar noon, daytime only
PQ = 1.2                    # photosynthetic quotient (O2/C)
RQ = 1.0                    # respiratory quotient (C/O2)

# Calibration parameters (to be optimized)
# e = ER:GPP (dimensionless), typical for Ca-HCO3 lakes ~0.6–0.8
E_BOUNDS = (0.4, 1.05)
# k600 (cm/hr), typical lake range ~2–30 cm/hr depending on wind, fetch
K600_BOUNDS = (2.0, 30.0)

# Carbon isotope fractionations (per mil, ε = δ_product - δ_substrate)
EPS_AIR_WATER = -1.0     # effective fractionation air CO2 -> DIC for invasion/evasion
EPS_PHOTO_CO2 = -23.0    # photosynthetic uptake from CO2(aq)
EPS_PHOTO_HCO3 = -12.0   # photosynthetic uptake from HCO3-
D13C_ORG = -28.0         # respired organic carbon δ13C

# Initial water chemistry (Ca-HCO3 type)
INITIAL_PH_GUESS = 8.2
ALKALINITY_MEQ_L = 3.0     # 2–4 meq/L typical
CA_MMOL_L = 2.0            # 1–3 mmol/L typical
INITIAL_DIC_MMOL_L = 2.5   # reasonable starting DIC for Ca-HCO3 lake
INITIAL_D13C_DIC = -7.5    # within observed range

# Time discretization
DT_HR = 1.0
N_STEPS = int(24 / DT_HR)

# Objective function choice
# "rmse" or "nrmse" (normalized by observed range)
FIT_METRIC = "rmse"

# Exports
EXPORT_DIR = "exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

# PHREEQC database path (optional override). If None, try phreeqpy default.
PHREEQC_DB_PATH = None  # e.g., "phreeqc.dat"

In [ ]:
def load_field_data(csv_path=None):
    """
    Expected CSV columns (minimum):
      - time_hour (0–24) or datetime
      - d13C_DIC_obs (per mil)

    If csv_path is None, generate a synthetic diel dataset in the observed summer range (-6.7 to -8.2).
    """
    if csv_path is None:
        t = np.arange(0, 24, 2)  # every 2 hours example
        # synthetic diel: midday enrichment, night depletion
        d = -7.7 + 0.6*np.sin(2*np.pi*(t-6)/24)  # range approx -8.3 to -7.1
        d = np.clip(d, -8.2, -6.7)
        df = pd.DataFrame({"time_hour": t, "d13C_DIC_obs": d})
        return df

    df = pd.read_csv(csv_path)
    if "time_hour" not in df.columns:
        raise ValueError("CSV must include 'time_hour' (0–24).")
    if "d13C_DIC_obs" not in df.columns:
        raise ValueError("CSV must include 'd13C_DIC_obs' (per mil).")
    return df.sort_values("time_hour").reset_index(drop=True)


field_df = load_field_data(csv_path=None)
field_df

In [ ]:
def calculate_sunrise_sunset(latitude, longitude, date_):
    """
    Returns (sunrise_dt, sunset_dt, solar_noon_dt, daylight_hours) in UTC by default.
    """
    loc = LocationInfo(name=SITE_NAME, region="NA", timezone=TIMEZONE_NAME,
                       latitude=latitude, longitude=longitude)
    s = sun(loc.observer, date=date_, tzinfo=timezone.utc if TIMEZONE_NAME.upper()=="UTC" else None)
    sunrise = s["sunrise"]
    sunset = s["sunset"]
    noon = s["noon"]
    daylight_hours = (sunset - sunrise).total_seconds() / 3600.0
    return sunrise, sunset, noon, daylight_hours


def calculate_gpp(hour_dt, sunrise, sunset, gpp_max):
    """
    Daytime-only smooth sinusoid:
      gpp = gpp_max * sin(pi * (t - sunrise)/daylen),  t in [sunrise, sunset]
    Outside daylight: exactly 0.
    """
    if hour_dt < sunrise or hour_dt > sunset:
        return 0.0
    daylen = (sunset - sunrise).total_seconds()
    if daylen <= 0:
        return 0.0
    x = (hour_dt - sunrise).total_seconds() / daylen  # 0..1
    # Sinusoid with 0 at edges, peak at x=0.5
    return float(gpp_max * np.sin(np.pi * x))


sunrise_dt, sunset_dt, solar_noon_dt, daylight_hours = calculate_sunrise_sunset(
    LATITUDE, LONGITUDE, DATE0
)
sunrise_dt, sunset_dt, solar_noon_dt, daylight_hours

In [ ]:
def schmidt_co2_freshwater(Tc):
    """
    Schmidt number for CO2 in freshwater (approx; commonly used polynomial).
    Valid ~0–30°C.
    """
    T = Tc
    return 1911.1 - 118.11*T + 3.4527*T**2 - 0.04132*T**3


def calculate_k600(wind_speed, temperature):
    """
    Cole & Caraco (1998): k600 (cm/hr) = 2.07 + 0.215 * U10^1.7
    Apply Schmidt correction for CO2: k = k600 * (Sc/600)^(-n), n~0.5.
    Returns (k600_cm_hr, k_m_hr).
    """
    k600_cm_hr = 2.07 + 0.215 * (wind_speed ** 1.7)
    Sc = schmidt_co2_freshwater(temperature)
    n = 0.5
    k_cm_hr = k600_cm_hr * (Sc / 600.0) ** (-n)
    k_m_hr = k_cm_hr / 100.0
    return float(k600_cm_hr), float(k_m_hr)


def henry_k0_co2_mol_L_atm(Tc):
    """
    Henry solubility for CO2 in freshwater, mol/(L·atm).
    Weiss-style formulations exist; here a compact approximation is used.
    For calibration/relative dynamics, this is generally adequate.
    """
    T = Tc + 273.15
    # Simple empirical approximation (order-correct): decreases with T
    # If you need higher accuracy, replace with Weiss (1974).
    lnK0 = -58.0931 + 90.5069*(100.0/T) + 22.2940*np.log(T/100.0)
    K0 = np.exp(lnK0)  # mol/(kg·atm) ~ mol/(L·atm)
    return float(K0)

In [ ]:
R_STD_VPDB = 0.0112372  # 13C/12C

def d13c_to_R(d13c_permil, Rstd=R_STD_VPDB):
    return Rstd * (d13c_permil/1000.0 + 1.0)

def R_to_d13c(R, Rstd=R_STD_VPDB):
    return 1000.0 * (R/Rstd - 1.0)

def split_12_13(total_mol, d13c_permil):
    R = d13c_to_R(d13c_permil)
    n13 = total_mol * (R / (1.0 + R))
    n12 = total_mol * (1.0 / (1.0 + R))
    return n12, n13

def combine_12_13(n12, n13):
    total = n12 + n13
    R = n13 / max(n12, 1e-30)
    d = R_to_d13c(R)
    return total, d

def remove_carbon(n12, n13, mol_remove, d13c_removed):
    mol_remove = max(0.0, mol_remove)
    total, d_pool = combine_12_13(n12, n13)
    mol_remove = min(mol_remove, total)
    rrem = d13c_to_R(d13c_removed)
    n13rem = mol_remove * (rrem/(1+rrem))
    n12rem = mol_remove * (1/(1+rrem))
    return n12 - n12rem, n13 - n13rem

def add_carbon(n12, n13, mol_add, d13c_add):
    mol_add = max(0.0, mol_add)
    radd = d13c_to_R(d13c_add)
    n13add = mol_add * (radd/(1+radd))
    n12add = mol_add * (1/(1+radd))
    return n12 + n12add, n13 + n13add

In [ ]:
class PhreeqcRunner:
    """
    Minimal wrapper for PHREEQC speciation using phreeqpy IPhreeqc.
    Uses RunString with embedded template and extracts selected output.
    """
    def __init__(self, db_path=None):
        self.ok = False
        self.backend = None

        try:
            from phreeqpy.iphreeqc.phreeqc_dll import IPhreeqc
            self.backend = IPhreeqc()
            # Try load database
            if db_path is not None:
                err = self.backend.load_database(db_path)
            else:
                # phreeqpy often exposes a default database location; try common names
                # If this fails, set PHREEQC_DB_PATH in config.
                err = self.backend.load_database("phreeqc.dat")
            if err != 0:
                raise RuntimeError("PHREEQC database load returned nonzero error code.")
            self.ok = True
        except Exception as e:
            self.ok = False
            self.backend = None
            self.init_error = str(e)

    def run_phreeqc_step(self, temp_c, ph_guess, alk_meq_l, ca_mmol_l, dic_mmol_l):
        """
        Solve carbonate speciation given Alk + DIC + Ca at temperature.
        We let PHREEQC adjust pH to charge balance using 'pH <guess> charge'.
        Returns dict with pH, CO2(aq), HCO3-, CO3-- (umol/L), DIC (mmol/L).
        """
        if not self.ok:
            raise RuntimeError(f"PHREEQC not available: {getattr(self, 'init_error', 'unknown error')}")

        # Convert to PHREEQC units (mol/kgw ~ mol/L for dilute freshwater)
        alk_eq_l = alk_meq_l * 1e-3  # meq/L -> eq/L
        dic_mol_l = dic_mmol_l * 1e-3
        ca_mol_l = ca_mmol_l * 1e-3

        phreeqc = f"""
SOLUTION 1
    temp {temp_c:.3f}
    pH {ph_guess:.4f} charge
    units mol/l
    Ca {ca_mol_l:.8e}
    Alkalinity {alk_eq_l:.8e}
    C {dic_mol_l:.8e}

SELECTED_OUTPUT
    -reset false
    -high_precision true
USER_PUNCH
    -headings pH CO2_molL HCO3_molL CO3_molL DIC_molL
10 PUNCH -LA("H+")
20 PUNCH MOL("CO2")
30 PUNCH MOL("HCO3-")
40 PUNCH MOL("CO3-2")
50 PUNCH TOT("C")
END
"""
        self.backend.run_string(phreeqc)
        arr = self.backend.get_selected_output_array()
        # arr includes header row; last row is the computed solution
        # phreeqpy returns a list-of-lists
        header = arr[0]
        last = arr[-1]
        out = dict(zip(header, last))

        # Convert mol/L to umol/L etc.
        co2_umol_l = float(out["CO2_molL"]) * 1e6
        hco3_umol_l = float(out["HCO3_molL"]) * 1e6
        co3_umol_l = float(out["CO3_molL"]) * 1e6
        dic_mmol_l_out = float(out["DIC_molL"]) * 1e3

        return {
            "pH": float(out["pH"]),
            "CO2_umol_L": co2_umol_l,
            "HCO3_umol_L": hco3_umol_l,
            "CO3_umol_L": co3_umol_l,
            "DIC_mmol_L": dic_mmol_l_out,
        }


# Initialize PHREEQC
phr = PhreeqcRunner(db_path=PHREEQC_DB_PATH)
if not phr.ok:
    display(Markdown(f"### PHREEQC initialization failed\n`{phr.init_error}`\n\n"
                     "Set `PHREEQC_DB_PATH` to a valid database file path, or install phreeqpy correctly."))
else:
    display(Markdown("### PHREEQC ready"))

In [ ]:
def analytical_daily_gpp(gpp_max, daylight_hours):
    # integral_0^D gpp_max*sin(pi t/D) dt = gpp_max * D * (2/pi)
    return float(gpp_max * daylight_hours * (2.0/np.pi))  # mg O2/L/day


def mgO2Lhr_to_molC_total(mgO2_L_hr, lake_liters, pq=PQ):
    molO2_L = (mgO2_L_hr / 1000.0) / 32.0
    molC_L = molO2_L / pq
    return molC_L * lake_liters


def mgO2Lhr_resp_to_molC_total(mgO2_L_hr, lake_liters, rq=RQ):
    molO2_L = (mgO2_L_hr / 1000.0) / 32.0
    molC_L = molO2_L * rq
    return molC_L * lake_liters

In [ ]:
def simulate_diel(e_ratio, k600_cm_hr, return_full=True):
    """
    Forward model for 24 h.
    Calibrated parameters:
      e_ratio = ER:GPP (daily-integrated, dimensionless)
      k600_cm_hr = gas transfer velocity at Sc=600 (cm/hr), treated constant across day here.

    Returns:
      df_time (hourly) with pH, pCO2, GPP, ER, flux, speciation, DIC, d13C.
    """
    # Build time axis (UTC)
    t0 = datetime(DATE0.year, DATE0.month, DATE0.day, 0, 0, tzinfo=timezone.utc)
    times = [t0 + timedelta(hours=i*DT_HR) for i in range(N_STEPS)]

    # Daylight + GPP
    sunrise, sunset, noon, daylen_hr = calculate_sunrise_sunset(LATITUDE, LONGITUDE, DATE0)
    gpp_series = np.array([calculate_gpp(t, sunrise, sunset, GPP_MAX_MG_O2_L_HR) for t in times], dtype=float)

    # Quality check: exactly zero at night
    for t, g in zip(times, gpp_series):
        if (t < sunrise or t > sunset) and g != 0.0:
            raise AssertionError("Nighttime GPP must be exactly 0.0")

    # ER computed from *analytical* daily GPP for stability, then distributed evenly
    gpp_daily_mgO2_L_d = analytical_daily_gpp(GPP_MAX_MG_O2_L_HR, daylen_hr)
    er_daily_mgO2_L_d = e_ratio * gpp_daily_mgO2_L_d
    er_hourly_mgO2_L_hr = er_daily_mgO2_L_d / 24.0
    er_series = np.full_like(gpp_series, er_hourly_mgO2_L_hr)

    # State: total DIC (whole-lake mol) and isotopes
    dic_mmol_L = INITIAL_DIC_MMOL_L
    dic_total_mol = dic_mmol_L * 1e-3 * LAKE_LITERS
    n12, n13 = split_12_13(dic_total_mol, INITIAL_D13C_DIC)
    ph_guess = INITIAL_PH_GUESS

    # Precompute k (CO2) from k600 using Schmidt correction
    Sc = schmidt_co2_freshwater(WATER_TEMPERATURE_C)
    n_sc = 0.5
    k_co2_cm_hr = k600_cm_hr * (Sc/600.0)**(-n_sc)
    k_co2_m_hr = k_co2_cm_hr / 100.0

    # Henry
    K0 = henry_k0_co2_mol_L_atm(WATER_TEMPERATURE_C)
    pco2_atm_atm = ATM_PCO2_UATM * 1e-6
    co2_eq_mol_L = K0 * pco2_atm_atm

    rows = []

    for i, t in enumerate(times):
        # --- Speciation at start-of-hour using current DIC ---
        dic_mmol_L = (n12 + n13) / LAKE_LITERS * 1e3  # mol->mmol/L
        spec = phr.run_phreeqc_step(
            temp_c=WATER_TEMPERATURE_C,
            ph_guess=ph_guess,
            alk_meq_l=ALKALINITY_MEQ_L,
            ca_mmol_l=CA_MMOL_L,
            dic_mmol_l=dic_mmol_L
        )
        ph_guess = spec["pH"]

        # Speciation-based uptake weighting proxy
        co2 = max(spec["CO2_umol_L"], 0.0)
        hco3 = max(spec["HCO3_umol_L"], 0.0)
        denom = max(co2 + hco3, 1e-12)
        fco2 = co2 / denom
        fhco3 = hco3 / denom
        eps_photo = fco2*EPS_PHOTO_CO2 + fhco3*EPS_PHOTO_HCO3

        # Compute pCO2 from CO2(aq) and Henry
        co2_mol_L = spec["CO2_umol_L"] * 1e-6
        pco2_water_atm = co2_mol_L / max(K0, 1e-30)
        pco2_water_uatm = pco2_water_atm * 1e6

        # --- Metabolism fluxes (whole-lake mol C per hour) ---
        gpp_mgO2_L_hr = gpp_series[i]
        er_mgO2_L_hr = er_series[i]

        molC_fix = mgO2Lhr_to_molC_total(gpp_mgO2_L_hr, LAKE_LITERS, pq=PQ) * DT_HR
        molC_resp = mgO2Lhr_resp_to_molC_total(er_mgO2_L_hr, LAKE_LITERS, rq=RQ) * DT_HR

        # --- Gas exchange (mol/m2/hr, positive evasion) ---
        # Use Cw - Ceq in mol/L, convert to mol/m3 by *1000
        Cw_mol_m3 = co2_mol_L * 1000.0
        Ceq_mol_m3 = co2_eq_mol_L * 1000.0
        F_mol_m2_hr = k_co2_m_hr * (Cw_mol_m3 - Ceq_mol_m3)  # + = evasion
        molCO2_gas_total = -F_mol_m2_hr * LAKE_AREA_M2 * DT_HR  # + adds to water, - removes from water

        # --- Isotope mass balance update (Python) ---
        total_mol, d13c_pool = combine_12_13(n12, n13)

        # Photosynthesis removes carbon with fractionation
        if molC_fix > 0:
            d13c_removed = d13c_pool + eps_photo
            n12, n13 = remove_carbon(n12, n13, molC_fix, d13c_removed)

        # Respiration adds carbon with δorg
        if molC_resp > 0:
            n12, n13 = add_carbon(n12, n13, molC_resp, D13C_ORG)

        # Gas exchange: invasion adds with (atm + eps); evasion removes with (pool + eps)
        if molCO2_gas_total > 0:
            n12, n13 = add_carbon(n12, n13, molCO2_gas_total, ATM_D13C_CO2 + EPS_AIR_WATER)
        elif molCO2_gas_total < 0:
            d13c_removed = d13c_pool + EPS_AIR_WATER
            n12, n13 = remove_carbon(n12, n13, abs(molCO2_gas_total), d13c_removed)

        # Post-update diagnostics
        total_mol, d13c_pool = combine_12_13(n12, n13)
        dic_mmol_L_next = total_mol / LAKE_LITERS * 1e3

        # Safety checks
        if total_mol < 0:
            raise RuntimeError("Negative DIC occurred (mass balance bug).")
        if (d13c_pool < -20) or (d13c_pool > 10):
            # Not fatal, but likely parameterization or initial conditions issue
            pass

        rows.append({
            "time": t,
            "hour": i*DT_HR,
            "GPP_mgO2_L_hr": gpp_mgO2_L_hr,
            "ER_mgO2_L_hr": er_mgO2_L_hr,
            "eps_photo_permil": eps_photo,
            "k600_cm_hr": k600_cm_hr,
            "kCO2_cm_hr": k_co2_cm_hr,
            "pH": spec["pH"],
            "CO2_flux_mol_m2_hr": F_mol_m2_hr,
            "molCO2_gas_total": molCO2_gas_total,
            "CO2_umol_L": spec["CO2_umol_L"],
            "HCO3_umol_L": spec["HCO3_umol_L"],
            "CO3_umol_L": spec["CO3_umol_L"],
            "pCO2_uatm": pco2_water_uatm,
            "DIC_mmol_L": spec["DIC_mmol_L"],
            "DIC_mmol_L_state": dic_mmol_L_next,
            "d13C_DIC_permil": d13c_pool,
        })

    df = pd.DataFrame(rows)
    return df


# Quick smoke test
if phr.ok:
    test_df = simulate_diel(e_ratio=0.7, k600_cm_hr=10.0)
    test_df.head()

In [ ]:
def interpolate_model_to_obs(model_df, obs_df):
    # linear interpolation on hour
    y = np.interp(obs_df["time_hour"].values, model_df["hour"].values, model_df["d13C_DIC_permil"].values)
    return y

def goodness_of_fit(model_at_obs, obs):
    resid = model_at_obs - obs
    rmse = float(np.sqrt(np.mean(resid**2)))
    if FIT_METRIC == "rmse":
        return rmse
    elif FIT_METRIC == "nrmse":
        rng = float(obs.max() - obs.min())
        return rmse / (rng if rng > 0 else 1.0)
    else:
        raise ValueError("Unknown FIT_METRIC")

def sweep_parameters(obs_df, e_grid, k600_grid):
    Z = np.zeros((len(e_grid), len(k600_grid)))
    for i, e in enumerate(e_grid):
        for j, k600 in enumerate(k600_grid):
            m = simulate_diel(e_ratio=e, k600_cm_hr=k600)
            m_obs = interpolate_model_to_obs(m, obs_df)
            Z[i, j] = goodness_of_fit(m_obs, obs_df["d13C_DIC_obs"].values)
    return Z


# Define sweep ranges (recommended)
e_vals = np.linspace(E_BOUNDS[0], E_BOUNDS[1], 26)          # ~0.4–1.05
k600_vals = np.linspace(K600_BOUNDS[0], K600_BOUNDS[1], 29) # ~2–30 cm/hr

if phr.ok:
    Z = sweep_parameters(field_df, e_vals, k600_vals)
    Z.min(), Z.max()

In [ ]:
if phr.ok:
    E, K = np.meshgrid(k600_vals, e_vals)  # note axes
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    cs = ax.contourf(K, E, Z, levels=20, cmap="viridis")
    cbar = fig.colorbar(cs, ax=ax)
    cbar.set_label(f"{FIT_METRIC.upper()} (‰)")

    # Mark best
    idx = np.unravel_index(np.argmin(Z), Z.shape)
    e_best_sweep = e_vals[idx[0]]
    k_best_sweep = k600_vals[idx[1]]
    ax.plot(k_best_sweep, e_best_sweep, "ro", label="Best (grid)")

    ax.set_xlabel("k600 (cm/hr)")
    ax.set_ylabel("e = ER:GPP (-)")
    ax.set_title("Parameter sweep goodness-of-fit (δ¹³C-DIC)")
    ax.legend()
    plt.show()

    e_best_sweep, k_best_sweep, Z[idx]

In [ ]:
def residuals_theta(theta, obs_df):
    # theta = [e, k600]
    e, k600 = theta
    m = simulate_diel(e_ratio=e, k600_cm_hr=k600)
    m_obs = interpolate_model_to_obs(m, obs_df)
    return (m_obs - obs_df["d13C_DIC_obs"].values)

if phr.ok:
    x0 = np.array([e_best_sweep, k_best_sweep], dtype=float)
    lb = np.array([E_BOUNDS[0], K600_BOUNDS[0]], dtype=float)
    ub = np.array([E_BOUNDS[1], K600_BOUNDS[1]], dtype=float)

    res = least_squares(
        residuals_theta,
        x0=x0,
        bounds=(lb, ub),
        args=(field_df,),
        method="trf",
        ftol=1e-10, xtol=1e-10, gtol=1e-10,
    )

    e_hat, k600_hat = res.x
    rmse_hat = np.sqrt(np.mean(res.fun**2))
    e_hat, k600_hat, rmse_hat

In [ ]:
def approx_param_covariance(lsq_result):
    """
    Approximate covariance from Jacobian at solution:
      cov ≈ s^2 (J^T J)^-1, s^2 = SSE/(n-p)
    """
    J = lsq_result.jac
    r = lsq_result.fun
    n = len(r)
    p = J.shape[1]
    s2 = (r @ r) / max(n - p, 1)
    JTJ = J.T @ J
    cov = s2 * np.linalg.inv(JTJ)
    return cov

if phr.ok:
    cov = approx_param_covariance(res)
    se = np.sqrt(np.diag(cov))
    display(pd.DataFrame({"param": ["e", "k600_cm_hr"], "estimate": [e_hat, k600_hat], "SE": se}))

    # Monte Carlo ensemble
    N_ENSEMBLE = 250
    thetas = multivariate_normal(mean=res.x, cov=cov, allow_singular=True).rvs(size=N_ENSEMBLE, random_state=RNG)
    # enforce bounds (clip)
    thetas[:, 0] = np.clip(thetas[:, 0], *E_BOUNDS)
    thetas[:, 1] = np.clip(thetas[:, 1], *K600_BOUNDS)

    base = simulate_diel(e_hat, k600_hat)

    # Run ensemble
    ens = []
    for th in thetas:
        ens.append(simulate_diel(th[0], th[1])[[
            "hour","pH","pCO2_uatm","GPP_mgO2_L_hr","ER_mgO2_L_hr","CO2_flux_mol_m2_hr",
            "CO2_umol_L","HCO3_umol_L","CO3_umol_L","DIC_mmol_L_state","d13C_DIC_permil"
        ]].assign(e=th[0], k600=th[1]))
    ens_df = pd.concat(ens, ignore_index=True)

    def envelope(var):
        g = ens_df.groupby("hour")[var]
        return pd.DataFrame({
            "hour": g.mean().index.values,
            f"{var}_lo": g.quantile(0.025).values,
            f"{var}_hi": g.quantile(0.975).values,
        })

    env = envelope("d13C_DIC_permil")
    env.head()

In [ ]:
def plot_with_envelope(ax, x, y, lo, hi, label, color):
    ax.plot(x, y, color=color, lw=2, label=label)
    ax.fill_between(x, lo, hi, color=color, alpha=0.2, linewidth=0)

if phr.ok:
    model = base.copy()

    # Envelopes for multiple vars
    env_vars = ["pH","pCO2_uatm","CO2_flux_mol_m2_hr","CO2_umol_L","HCO3_umol_L","CO3_umol_L","DIC_mmol_L_state","d13C_DIC_permil"]
    env_map = {v: envelope(v) for v in env_vars}

    fig, axes = plt.subplots(4, 2, figsize=(12, 12), sharex=True)
    axes = axes.ravel()

    # 1 pH
    v="pH"; e=env_map[v]
    plot_with_envelope(axes[0], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "Modeled pH", "tab:blue")
    axes[0].set_ylabel("pH (-)")
    axes[0].legend()

    # 2 pCO2
    v="pCO2_uatm"; e=env_map[v]
    plot_with_envelope(axes[1], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "Modeled pCO₂", "tab:orange")
    axes[1].set_ylabel("pCO₂ (µatm)")
    axes[1].legend()

    # 3 GPP/ER (as mg O2/L/hr; you can convert to mmol C/m2/hr if desired)
    axes[2].plot(model["hour"], model["GPP_mgO2_L_hr"], lw=2, label="GPP", color="tab:green")
    axes[2].plot(model["hour"], model["ER_mgO2_L_hr"], lw=2, label="ER", color="tab:red")
    axes[2].set_ylabel("mg O₂ L⁻¹ h⁻¹")
    axes[2].legend()

    # 4 CO2 flux
    v="CO2_flux_mol_m2_hr"; e=env_map[v]
    plot_with_envelope(axes[3], model["hour"], 1e3*model[v], 1e3*e[f"{v}_lo"], 1e3*e[f"{v}_hi"], "CO₂ flux", "tab:purple")
    axes[3].axhline(0, color="k", lw=1)
    axes[3].set_ylabel("CO₂ flux (mmol m⁻² h⁻¹)\n(+ evasion)")
    axes[3].legend()

    # 5 CO2(aq)
    v="CO2_umol_L"; e=env_map[v]
    plot_with_envelope(axes[4], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "CO₂(aq)", "tab:cyan")
    axes[4].set_ylabel("CO₂(aq) (µmol L⁻¹)")
    axes[4].legend()

    # 6 HCO3-
    v="HCO3_umol_L"; e=env_map[v]
    plot_with_envelope(axes[5], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "HCO₃⁻", "tab:gray")
    axes[5].set_ylabel("HCO₃⁻ (µmol L⁻¹)")
    axes[5].legend()

    # 7 CO3--
    v="CO3_umol_L"; e=env_map[v]
    plot_with_envelope(axes[6], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "CO₃²⁻", "tab:brown")
    axes[6].set_ylabel("CO₃²⁻ (µmol L⁻¹)")
    axes[6].legend()

    # 8 δ13C(DIC) modeled vs observed
    v="d13C_DIC_permil"; e=env_map[v]
    plot_with_envelope(axes[7], model["hour"], model[v], e[f"{v}_lo"], e[f"{v}_hi"], "Modeled δ¹³C(DIC)", "tab:blue")
    axes[7].scatter(field_df["time_hour"], field_df["d13C_DIC_obs"], c="k", s=35, label="Observed")
    axes[7].set_ylabel("δ¹³C(DIC) (‰)")
    axes[7].legend()

    for ax in axes:
        ax.set_xlim(0, 23)
        ax.set_xlabel("Hour (UTC)")

    fig.suptitle(f"Diel model outputs with 95% envelopes (e={e_hat:.3f}, k600={k600_hat:.2f} cm/hr)", y=0.92)
    plt.tight_layout()
    plt.show()

In [ ]:
def interactive_view(e_val, k600_val):
    clear_output(wait=True)
    m = simulate_diel(e_val, k600_val)
    m_obs = interpolate_model_to_obs(m, field_df)
    rmse = goodness_of_fit(m_obs, field_df["d13C_DIC_obs"].values)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(m["hour"], m["d13C_DIC_permil"], lw=2, label="Modeled")
    ax.scatter(field_df["time_hour"], field_df["d13C_DIC_obs"], c="k", label="Observed")
    ax.set_xlabel("Hour")
    ax.set_ylabel("δ¹³C(DIC) (‰)")
    ax.set_title(f"Interactive fit | e={e_val:.3f}, k600={k600_val:.2f} cm/hr | {FIT_METRIC.upper()}={rmse:.3f}‰")
    ax.legend()
    plt.show()

e_slider = widgets.FloatSlider(value=float(e_hat) if phr.ok else 0.7, min=E_BOUNDS[0], max=E_BOUNDS[1], step=0.01, description="e=ER:GPP")
k_slider = widgets.FloatSlider(value=float(k600_hat) if phr.ok else 10.0, min=K600_BOUNDS[0], max=K600_BOUNDS[1], step=0.5, description="k600 (cm/hr)")

ui = widgets.HBox([e_slider, k_slider])
out = widgets.interactive_output(interactive_view, {"e_val": e_slider, "k600_val": k_slider})
display(ui, out)

In [ ]:
if phr.ok:
    # Numerical integration from hourly series
    sunrise, sunset, noon, daylen_hr = calculate_sunrise_sunset(LATITUDE, LONGITUDE, DATE0)
    t0 = datetime(DATE0.year, DATE0.month, DATE0.day, 0, 0, tzinfo=timezone.utc)
    times = [t0 + timedelta(hours=i*DT_HR) for i in range(N_STEPS)]
    gpp_hr = np.array([calculate_gpp(t, sunrise, sunset, GPP_MAX_MG_O2_L_HR) for t in times])
    gpp_num = float(np.sum(gpp_hr) * DT_HR)  # mg O2/L/day (rectangular)
    gpp_ana = analytical_daily_gpp(GPP_MAX_MG_O2_L_HR, daylen_hr)
    pct = 100.0 * (gpp_num - gpp_ana) / gpp_ana

    print(f"Daylight hours: {daylen_hr:.2f} h")
    print(f"Analytical daily GPP: {gpp_ana:.3f} mg O2/L/d")
    print(f"Numerical daily GPP:  {gpp_num:.3f} mg O2/L/d")
    print(f"Percent difference:   {pct:.2f} %")

In [ ]:
def timestamp_tag():
    return datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

if phr.ok:
    tag = timestamp_tag()

    # Rebuild the multi-panel plot and save as SVG
    figpath = os.path.join(EXPORT_DIR, f"diel_outputs_{tag}.svg")

    # Create a single SVG figure (reuse code quickly)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(base["hour"], base["d13C_DIC_permil"], lw=2, label="Modeled")
    ax.scatter(field_df["time_hour"], field_df["d13C_DIC_obs"], c="k", label="Observed")
    ax.set_xlabel("Hour")
    ax.set_ylabel("δ¹³C(DIC) (‰)")
    ax.legend()
    ax.set_title("δ¹³C(DIC) modeled vs observed")
    fig.tight_layout()
    fig.savefig(figpath, format="svg")
    plt.show()
    print("Saved:", figpath)

    # Build export dataframe with envelopes for each variable
    export_df = base.copy()

    # Merge observed
    export_df["d13C_DIC_obs"] = np.interp(export_df["hour"], field_df["time_hour"], field_df["d13C_DIC_obs"])

    # Add envelopes
    for v in ["pH","pCO2_uatm","CO2_flux_mol_m2_hr","CO2_umol_L","HCO3_umol_L","CO3_umol_L","DIC_mmol_L_state","d13C_DIC_permil"]:
        e = envelope(v)
        export_df = export_df.merge(e, on="hour", how="left")

    # Unit convenience columns
    export_df["CO2_flux_mmol_m2_hr"] = 1e3 * export_df["CO2_flux_mol_m2_hr"]

    csvpath = os.path.join(EXPORT_DIR, f"diel_timeseries_{tag}.csv")
    export_df.to_csv(csvpath, index=False)
    print("Saved:", csvpath)

## Methods paragraph (auto-generated)

In [ ]:
def methods_paragraph(e_hat, k600_hat, cov, metric_name):
    se = np.sqrt(np.diag(cov))
    txt = f"""
We calibrated a diel carbon isotope mass-balance model to field observations of δ¹³C(DIC) over a 24 h cycle by optimizing two parameters: (i) e = ER:GPP, the ratio of daily ecosystem respiration to daily gross primary production, and (ii) k600 (cm h⁻¹), the air–water CO₂ gas transfer velocity normalized to a Schmidt number of 600. Hourly GPP was prescribed as a daytime-only sinusoid peaking at solar noon, with sunrise and sunset computed from site latitude/longitude and sampling date; ER was applied as a constant rate over 24 h such that daily ER = e·GPP. Carbonate system speciation (pH, CO₂(aq), HCO₃⁻, CO₃²⁻) was solved at each time step using PHREEQC (IPhreeqc via phreeqpy) with a calcium–bicarbonate solution constrained by measured alkalinity and modeled DIC. δ¹³C(DIC) dynamics were computed by explicit ¹²C/¹³C mass balance including fractionation during photosynthetic uptake (CO₂ vs HCO₃⁻ weighted by modeled speciation), respiratory DIC addition (δ¹³Cₒᵣg assumed constant), and air–water exchange (ε ≈ {EPS_AIR_WATER:.1f}‰; atmospheric δ¹³C ≈ {ATM_D13C_CO2:.1f}‰). Parameters were estimated using bounded nonlinear least squares minimizing {metric_name.upper()} between modeled and observed δ¹³C(DIC). Uncertainty was quantified from the Jacobian-based parameter covariance and propagated to diel predictions using Monte Carlo sampling to produce 95% confidence envelopes. Analyses were performed in Python (numpy, pandas, scipy, matplotlib, ipywidgets) coupled to PHREEQC through phreeqpy.
""".strip()
    return txt

if phr.ok:
    display(Markdown(methods_paragraph(e_hat, k600_hat, cov, FIT_METRIC)))

## Binder-ready configuration

### Option A: `requirements.txt`
```text
numpy
pandas
matplotlib
scipy
ipywidgets
astral
phreeqpy


---

### 22) User documentation (Markdown cell)

```markdown
## Using your own field data

### Expected CSV format
Minimum columns:
- `time_hour` : hours since midnight (0–24)
- `d13C_DIC_obs` : observed δ¹³C(DIC) in ‰

Optional columns:
- replicate IDs, uncertainties, temperature, wind speed (you can extend the model to ingest time-varying forcing)

Load with:
```python
field_df = load_field_data("my_diel_d13c.csv")


---

## Brief interpretation (3–4 sentences)

With typical parameterizations, modeled δ¹³C(DIC) becomes **enriched during daylight** as photosynthesis preferentially removes ^12C (with effective ε weighted by CO₂ vs HCO₃⁻ uptake, which is controlled by pH/speciation). During night, δ¹³C(DIC) **decreases** as respiration adds ^13C-depleted DIC (δ¹³Cₒᵣg ≈ −28‰) while GPP is exactly zero. Air–water gas exchange moderates the diel amplitude by removing CO₂ when water pCO₂ exceeds atmospheric and by pulling the dissolved pool toward the atmospheric isotopic signature (δ¹³C ≈ −8‰ with ε ≈ −1‰). The calibration primarily trades off **e** (controls isotopic depletion from respired inputs) against **k600** (controls damping and mean-state drift via exchange).

---

### One clarification needed (so I can tailor the notebook more tightly)
For your “**e?** parameter”, do you want it defined as **ER:GPP (daily integrated)** (what I implemented), or as an **ER rate scalar** (e.g., mmol C m⁻² h⁻¹) independent of GPP? If you specify which convention your field team uses, I can adjust the parameterization and units (including converting GPP/ER to mmol C m⁻² h⁻¹ and g C m⁻² d⁻¹ throughout).